# road_model 使用样例与可视化

演示内容：
1. 固定配置生成道路（直道 → 弯道 → 路口 → 分道 → 弯道 → 合道 → 直道）
2. 随机生成道路
3. 单车道 vs 双车道对比
4. `get_nearest_centerline_info` / `get_road_segment_ahead` 查询可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sim_env.road_model import (
    RoadGenerationConfig,
    RoadModel,
    RoadSegmentType,
    SegmentSpec,
    plot_road
)

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

## 示例 1：固定配置生成道路

使用 `generate_fixed()` 按指定片段序列生成道路：直道 → 弯道 → 路口 → 分道 → 弯道 → 合道 → 直道。
红点标注每段起始位置。

In [ ]:
segments = [
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 60, "num_lanes": 5}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 20}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 20, "angle_deg": 190}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 20}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 30, "angle_deg": -160, "num_lanes": 5}),
    # SegmentSpec(RoadSegmentType.INTERSECTION, {"size": 15}),
    # SegmentSpec(RoadSegmentType.SPLIT, {"length": 25, "angle_deg": 20}),
    # SegmentSpec(RoadSegmentType.CURVE, {"radius": 50, "angle_deg": -45}),
    # SegmentSpec(RoadSegmentType.MERGE, {"length": 20, "angle_deg": 15}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 20}),
]

road = RoadModel(RoadGenerationConfig(num_lanes=3, lane_width=3.5))
geo = road.generate_fixed(segments)

fig, ax = plt.subplots(figsize=(14, 6))
plot_road(ax, geo, title=f"固定配置道路 (7段, 总长 {geo.total_length:.0f}m, 双车道)")

fig.tight_layout()
plt.show()

In [ ]:
geo.road_segments[0].centerline[-1],geo.road_segments[1].centerline[0]
geo.road_segments[1].centerline[-1],geo.road_segments[2].centerline[0]
geo.road_segments[2].centerline[-1],geo.road_segments[3].centerline[0]

geo.road_segments[0].left_boundary[-1],geo.road_segments[1].left_boundary[0]
geo.road_segments[1].left_boundary[-1],geo.road_segments[2].left_boundary[0]
# geo.road_segments[2].left_boundary[-1],geo.road_segments[3].left_boundary[0]


## 示例 2：随机生成道路

使用 `generate_random()` 配合不同 seed 随机生成道路。
通过 `segment_weights` 控制各类型出现概率（弯道权重更高）。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, seed in zip(axes.flat, [42, 123, 7, 2024]):
    rm = RoadModel(RoadGenerationConfig(
        num_random_segments=8, num_lanes=1, lane_width=3.5,
        segment_weights={
            RoadSegmentType.STRAIGHT: 3.0,
            RoadSegmentType.CURVE: 4.0,
            RoadSegmentType.INTERSECTION: 1.0,
            RoadSegmentType.SPLIT: 1.0,
            RoadSegmentType.MERGE: 1.0,
        },
    ))
    g = rm.generate_random(seed=seed)
    plot_road(ax, g, title=f"随机道路 seed={seed} ({g.total_length:.0f}m)", show_dividers=False)

fig.tight_layout()
plt.show()

## 示例 3：单车道 vs 双车道

同一路线，`num_lanes=1` 和 `num_lanes=2` 的对比。双车道时自动生成车道分隔线。

In [ ]:
lane_segments = [
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 40}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 35, "angle_deg": 90}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 6))

for ax, n_lanes in zip(axes, [1, 2, 3]):
    rm = RoadModel(RoadGenerationConfig(num_lanes=n_lanes, lane_width=3.5))
    g = rm.generate_fixed(lane_segments)
    plot_road(ax, g, title=f"{n_lanes}车道 (路宽 {n_lanes * 3.5}m)")

fig.tight_layout()
plt.show()

## 示例 4：查询接口可视化

演示两个查询 API：
- `get_nearest_centerline_info(x, y)` → 横向偏移、道路朝向、纵向进度
- `get_road_segment_ahead(x, y, num_points)` → 前方道路的中心线/左边界/右边界绝对坐标

模拟车辆偏离中心线 2m 的场景。

In [ ]:
# 复用示例1的 road 对象
geo = road.geometry

# 将所有片段的中心线连接起来
all_centerline = np.concatenate([seg.centerline for seg in geo.road_segments], axis=0)

# 在中心线 1/4 处模拟一个偏离 2m 的车辆
idx_query = len(all_centerline) // 4
ref_pt = all_centerline[idx_query]
if idx_query < len(all_centerline) - 1:
    tangent = all_centerline[idx_query + 1] - all_centerline[idx_query]
else:
    tangent = all_centerline[idx_query] - all_centerline[idx_query - 1]
heading = np.arctan2(tangent[1], tangent[0])
normal = np.array([-np.sin(heading), np.cos(heading)])

car_x = ref_pt[0] + normal[0] * 2.0
car_y = ref_pt[1] + normal[1] * 2.0

# 调用查询接口
lateral, road_heading, progress = road.get_nearest_centerline_info(car_x, car_y)
cl_pts, lb_pts, rb_pts, div_pts, nearest_idx, actual_num_points, actual_length_m, road_types = road.get_road_segment_ahead(car_x, car_y, num_points=80)

print(f"lateral_offset = {lateral:.2f} m")
print(f"road_heading   = {np.degrees(road_heading):.1f}°")
print(f"progress       = {progress:.3f}")
print(f"前方道路段形状:  centerline={cl_pts.shape}, left={lb_pts.shape}, right={rb_pts.shape}, dividers={len(div_pts)}条")
print(f"实际有效点数:    {actual_num_points}")
print(f"实际有效长度:    {actual_length_m:.2f} m")
print(f"前方道路类型:    {[t.value for t in road_types]}")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_road(ax, geo, title="查询接口演示", show_dividers=True)

# 车辆位置
ax.plot(car_x, car_y, "g*", ms=12, zorder=5, label="车辆位置")


# 前方道路段高亮
ax.plot(cl_pts[:, 0], cl_pts[:, 1], "c-", lw=3, alpha=0.7, label="前方中心线段")
ax.plot(lb_pts[:, 0], lb_pts[:, 1], "m-", lw=2, alpha=0.7, label="前方左边界段")
ax.plot(rb_pts[:, 0], rb_pts[:, 1], "m-", lw=2, alpha=0.7, label="前方右边界段")
for i, dp in enumerate(div_pts):
    label = "前方车道线段" if i == 0 else None
    ax.plot(dp[:, 0], dp[:, 1], "lime", lw=2, ls="--", alpha=0.7, label=label)

# 信息标注
info_text = (
    f"lateral_offset = {lateral:.2f} m\n"
    f"road_heading = {np.degrees(road_heading):.1f}°\n"
    f"progress = {progress:.3f}"
)
ax.text(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", color="cyan",
        bbox=dict(boxstyle="round", facecolor="black", alpha=0.7))

ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

## 示例 5：分别展示每种道路类型

将 5 种道路片段（直道、弯道、路口、分道、合道）各自单独生成并绘制，直观对比几何形状差异。

In [ ]:
# 每种道路类型的典型参数
segment_configs = {
    "STRAIGHT（直道）": SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 80}),
    "CURVE（弯道·左转60°）": SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
    "INTERSECTION（路口）": SegmentSpec(RoadSegmentType.INTERSECTION, {"size": 20}),
    "SPLIT（分道）": SegmentSpec(RoadSegmentType.SPLIT, {"length": 30, "angle_deg": 20}),
    "MERGE（合道）": SegmentSpec(RoadSegmentType.MERGE, {"length": 30, "angle_deg": 15}),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, spec) in zip(axes, segment_configs.items()):
    rm = RoadModel(RoadGenerationConfig(num_lanes=3, lane_width=3.5))
    g = rm.generate_fixed([spec])
    plot_road(ax, g, title=name, show_dividers=True)
    # 标注起点和终点
    ax.plot(*g.road_segments[0].centerline[0], "go", ms=8, label="起点")
    ax.plot(*g.road_segments[-1].centerline[-1], "rs", ms=8, label="终点")
    ax.legend(fontsize=6, loc="lower right")

fig.suptitle("五种道路片段类型对比（双车道）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# 每种道路类型的典型参数
segment_configs = {
    "STRAIGHT（直道）": SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 80}),
    "CURVE（弯道·左转60°）": SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
    "INTERSECTION（路口）": SegmentSpec(RoadSegmentType.INTERSECTION, {"size": 20}),
    "SPLIT（分道）": SegmentSpec(RoadSegmentType.SPLIT, {"length": 30, "angle_deg": 20}),
    "MERGE（合道）": SegmentSpec(RoadSegmentType.MERGE, {"length": 30, "angle_deg": 15}),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, spec) in zip(axes, segment_configs.items()):
    rm = RoadModel(RoadGenerationConfig(num_lanes=1, lane_width=3.5))
    g = rm.generate_fixed([spec])
    plot_road(ax, g, title=name, show_dividers=True)
    # 标注起点和终点
    ax.plot(*g.road_segments[0].centerline[0], "go", ms=8, label="起点")
    ax.plot(*g.road_segments[-1].centerline[-1], "rs", ms=8, label="终点")
    ax.legend(fontsize=6, loc="lower right")

fig.suptitle("五种道路片段类型对比（双车道）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 5：自动延长道路

演示 `extend_road` 功能：在当前道路末端追加新片段，保留历史道路。
- 传入 `segments` 参数则固定生成
- 不传参数则随机生成

In [ ]:
# 初始道路
cfg = RoadGenerationConfig(num_random_segments=4, num_lanes=1, lane_width=3.5)
rm = RoadModel(cfg)
rm.generate_random(seed=42)

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# 第 1 张：初始道路
plot_road(axes[0], rm.geometry, title='初始道路')

# 第 2 张：随机延长一次
rm.extend_road()
plot_road(axes[1], rm.geometry, title='随机延长 ×1')

# 第 3 张：再随机延长一次
rm.extend_road()
plot_road(axes[2], rm.geometry, title='随机延长 ×2')

# 第 4 张：用固定片段延长
fixed = [
    SegmentSpec(RoadSegmentType.STRAIGHT, {'length': 50}),
    SegmentSpec(RoadSegmentType.CURVE, {'radius': 30, 'angle_deg': 90}),
]
rm.extend_road(fixed)
plot_road(axes[3], rm.geometry, title='固定片段延长')

fig.suptitle('extend_road 演示：在末端追加新片段，保留历史道路', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()